# Notebook 08 — Machine Learning Models

This notebook builds supervised machine-learning baselines for S&P 500 next-day return prediction.

**Input**
`data/raw/sp500_1950_present.csv`

**Models**
- Logistic Regression
- Random Forest
- HistGradientBoosting
- XGBoost when available

**Target**
- Next-day return direction: `1` if next-day return > 0, otherwise `0`

**Methodology**
- Strictly chronological split
- All predictors are constructed from information available at or before time `t`
- No random train/test shuffling
- Feature scaling is fitted only on training data
- Classification metrics include accuracy, precision, recall, F1, ROC-AUC and directional accuracy
- Probability predictions are retained for later backtesting

This notebook is a research baseline. It does not claim predictive edge or deployability.


## 1. Imports

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")

print("Core imports loaded successfully.")

try:
    from xgboost import XGBClassifier
    xgb_available = True
    print("XGBoost available.")
except ImportError:
    XGBClassifier = None
    xgb_available = False
    print("XGBoost unavailable; the notebook will continue without it.")


## 2. Configuration and Paths

In [ ]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "data").exists():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/mnt/data/quant-trading-research"),
    ]
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            PROJECT_ROOT = candidate
            break

MASTER_PATH = PROJECT_ROOT / "data" / "raw" / "sp500_1950_present.csv"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
REPORT_DIR = PROJECT_ROOT / "reports" / "generated"

for path in [INTERIM_DIR, FIGURE_DIR, TABLE_DIR, REPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

EXPECTED_COLUMNS = [
    "Date", "Open", "High", "Low", "Close", "Adj.Close", "Volume"
]

TEST_FRACTION = 0.20

print(f"Master dataset: {MASTER_PATH}")


## 3. Load and Validate the Master Dataset

In [ ]:
if not MASTER_PATH.exists():
    raise FileNotFoundError(
        f"Master dataset not found: {MASTER_PATH}. "
        "Run the earlier notebooks first."
    )

df = pd.read_csv(MASTER_PATH, low_memory=False)

if list(df.columns) != EXPECTED_COLUMNS:
    raise ValueError(
        f"Unexpected schema. Expected {EXPECTED_COLUMNS}; "
        f"received {list(df.columns)}"
    )

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

for column in EXPECTED_COLUMNS[1:]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

if df["Date"].isna().any():
    raise ValueError("Invalid dates detected.")

if df["Date"].duplicated().any():
    raise ValueError("Duplicate dates detected.")

if df[["Open", "High", "Low", "Close", "Adj.Close", "Volume"]].isna().any().any():
    raise ValueError("Missing OHLCV values detected.")

print(f"Rows: {len(df):,}")
print(f"Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")


## 4. Build Strictly Backward-Looking Features

Every predictor below is calculated using current or historical observations only.

The target is created **after** feature construction and is shifted one day backward relative to the predictor row.


In [ ]:
df["return_1d"] = df["Close"].pct_change()
df["return_5d"] = df["Close"].pct_change(5)
df["return_21d"] = df["Close"].pct_change(21)
df["return_63d"] = df["Close"].pct_change(63)
df["return_126d"] = df["Close"].pct_change(126)
df["return_252d"] = df["Close"].pct_change(252)

df["volatility_5d"] = (
    df["return_1d"].rolling(5).std() * np.sqrt(252)
)

df["volatility_21d"] = (
    df["return_1d"].rolling(21).std() * np.sqrt(252)
)

df["volatility_63d"] = (
    df["return_1d"].rolling(63).std() * np.sqrt(252)
)

df["sma_20"] = df["Close"].rolling(20).mean()
df["sma_50"] = df["Close"].rolling(50).mean()
df["sma_200"] = df["Close"].rolling(200).mean()

df["price_to_sma_20"] = df["Close"] / df["sma_20"] - 1
df["price_to_sma_50"] = df["Close"] / df["sma_50"] - 1
df["price_to_sma_200"] = df["Close"] / df["sma_200"] - 1
df["sma_50_vs_sma_200"] = df["sma_50"] / df["sma_200"] - 1

df["range_pct"] = (
    (df["High"] - df["Low"]) / df["Close"]
)

df["intraday_return"] = (
    df["Close"] / df["Open"] - 1
)

df["overnight_return"] = (
    df["Open"] / df["Close"].shift(1) - 1
)

df["volume_ratio_20"] = (
    df["Volume"] / df["Volume"].rolling(20).mean()
)

df["volume_ratio_63"] = (
    df["Volume"] / df["Volume"].rolling(63).mean()
)

df["return_1d_lag1"] = df["return_1d"].shift(1)
df["return_1d_lag2"] = df["return_1d"].shift(2)
df["return_1d_lag3"] = df["return_1d"].shift(3)
df["return_1d_lag5"] = df["return_1d"].shift(5)
df["return_1d_lag10"] = df["return_1d"].shift(10)

df["volatility_21d_lag1"] = df["volatility_21d"].shift(1)
df["range_pct_lag1"] = df["range_pct"].shift(1)
df["volume_ratio_20_lag1"] = df["volume_ratio_20"].shift(1)


## 5. Create the Next-Day Direction Target

`target = 1` means the next trading day's close-to-close return is positive.

The target is never included among the predictors.


In [ ]:
df["next_day_return"] = (
    df["Close"].shift(-1) / df["Close"] - 1
)

df["target"] = (
    df["next_day_return"] > 0
).astype(int)

print(df[["Date", "Close", "next_day_return", "target"]].tail(10))


## 6. Define the Feature Set

In [ ]:
feature_columns = [
    "return_1d",
    "return_5d",
    "return_21d",
    "return_63d",
    "return_126d",
    "return_252d",
    "volatility_5d",
    "volatility_21d",
    "volatility_63d",
    "price_to_sma_20",
    "price_to_sma_50",
    "price_to_sma_200",
    "sma_50_vs_sma_200",
    "range_pct",
    "intraday_return",
    "overnight_return",
    "volume_ratio_20",
    "volume_ratio_63",
    "return_1d_lag1",
    "return_1d_lag2",
    "return_1d_lag3",
    "return_1d_lag5",
    "return_1d_lag10",
    "volatility_21d_lag1",
    "range_pct_lag1",
    "volume_ratio_20_lag1",
]

print(f"Number of features: {len(feature_columns)}")
for feature in feature_columns:
    print(f" - {feature}")


## 7. Remove Rows Without Sufficient Historical Features

Rows at the beginning of the history do not have enough lookback data for the longest features.

The final row has no next-day target and is removed.

No imputation is performed across the time boundary.


In [ ]:
model_df = df[
    ["Date", "Close", "next_day_return", "target"] + feature_columns
].copy()

model_df = model_df.dropna(
    subset=feature_columns + ["next_day_return", "target"]
).reset_index(drop=True)

print(f"Modeling rows: {len(model_df):,}")
print(f"Feature matrix shape: {model_df[feature_columns].shape}")
print(
    "Target distribution:",
    model_df["target"].value_counts(normalize=True).sort_index().to_dict()
)


## 8. Chronological Train/Test Split

In [ ]:
split_idx = int(len(model_df) * (1 - TEST_FRACTION))

train_df = model_df.iloc[:split_idx].copy()
test_df = model_df.iloc[split_idx:].copy()

X_train = train_df[feature_columns]
X_test = test_df[feature_columns]

y_train = train_df["target"]
y_test = test_df["target"]

print(f"Train rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")
print(f"Train end: {train_df['Date'].max().date()}")
print(f"Test start: {test_df['Date'].min().date()}")
print(f"Test end: {test_df['Date'].max().date()}")


## 9. Target Distribution by Split

In [ ]:
target_summary = pd.DataFrame({
    "train": y_train.value_counts(normalize=True).sort_index(),
    "test": y_test.value_counts(normalize=True).sort_index(),
}).fillna(0)

target_summary.index = ["Down/Flat", "Up"]

display(target_summary)

target_summary.to_csv(
    TABLE_DIR / "sp500_ml_target_distribution.csv"
)


## 10. Evaluation Function

In [ ]:
def evaluate_classifier(model_name, y_true, predictions, probabilities):
    metrics = {
        "model": model_name,
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_true, predictions, zero_division=0
        ),
        "roc_auc": (
            roc_auc_score(y_true, probabilities)
            if len(np.unique(y_true)) == 2
            else np.nan
        ),
    }

    return metrics


## 11. Model Definitions

All models are trained only on the chronological training period.

Logistic Regression uses scaling.

Tree models do not require feature scaling, but median imputation is retained as a defensive preprocessing step.


In [ ]:
models = {}

models["Logistic Regression"] = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            random_state=42,
            class_weight=None,
        ),
    ),
])

models["Random Forest"] = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    (
        "model",
        RandomForestClassifier(
            n_estimators=400,
            max_depth=8,
            min_samples_leaf=10,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1,
            class_weight=None,
        ),
    ),
])

models["HistGradientBoosting"] = Pipeline([
    (
        "model",
        HistGradientBoostingClassifier(
            max_iter=300,
            learning_rate=0.05,
            max_leaf_nodes=15,
            l2_regularization=1.0,
            random_state=42,
        ),
    ),
])

if xgb_available:
    models["XGBoost"] = XGBClassifier(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.80,
        colsample_bytree=0.80,
        min_child_weight=5,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    )

print("Models:")
for name in models:
    print(f" - {name}")


## 12. Train Models and Generate Test Predictions

In [ ]:
trained_models = {}
predictions = {}
probabilities = {}
model_metrics = []

for name, model in models.items():
    print(f"Training {name}...")

    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]

    trained_models[name] = model
    predictions[name] = pd.Series(
        pred,
        index=test_df.index,
        name="prediction"
    )
    probabilities[name] = pd.Series(
        prob,
        index=test_df.index,
        name="probability_up"
    )

    model_metrics.append(
        evaluate_classifier(
            name,
            y_test,
            pred,
            prob,
        )
    )

    print(f"Completed: {name}")

metrics = pd.DataFrame(model_metrics)

display(
    metrics.sort_values(
        "roc_auc",
        ascending=False
    ).reset_index(drop=True)
)


## 13. Save Model Comparison

In [ ]:
metrics = metrics.sort_values(
    ["roc_auc", "f1"],
    ascending=False
).reset_index(drop=True)

metrics.to_csv(
    TABLE_DIR / "sp500_machine_learning_model_comparison.csv",
    index=False
)

display(metrics)


## 14. Classification Reports

In [ ]:
classification_reports = {}

for name in trained_models:
    report = classification_report(
        y_test,
        predictions[name],
        target_names=["Down/Flat", "Up"],
        zero_division=0,
        output_dict=True,
    )

    classification_reports[name] = report

    print("=" * 72)
    print(name)
    print("=" * 72)
    print(
        classification_report(
            y_test,
            predictions[name],
            target_names=["Down/Flat", "Up"],
            zero_division=0,
        )
    )


## 15. Confusion Matrices

In [ ]:
for name in trained_models:
    cm = confusion_matrix(
        y_test,
        predictions[name]
    )

    print(f"{name}")
    display(
        pd.DataFrame(
            cm,
            index=["Actual Down/Flat", "Actual Up"],
            columns=["Predicted Down/Flat", "Predicted Up"],
        )
    )


## 16. ROC-AUC Comparison

In [ ]:
fig = plt.figure(figsize=(12, 6))

plt.bar(
    metrics["model"],
    metrics["roc_auc"]
)

plt.title("Machine Learning Model ROC-AUC")
plt.xlabel("Model")
plt.ylabel("ROC-AUC")
plt.xticks(rotation=20)
plt.grid(True, axis="y", alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_ml_model_roc_auc.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 17. F1 Comparison

In [ ]:
fig = plt.figure(figsize=(12, 6))

plt.bar(
    metrics["model"],
    metrics["f1"]
)

plt.title("Machine Learning Model F1 Score")
plt.xlabel("Model")
plt.ylabel("F1")
plt.xticks(rotation=20)
plt.grid(True, axis="y", alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_ml_model_f1.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 18. Probability Calibration View

Predicted probability bins are compared with realized positive-return frequency.

This is a descriptive calibration check on the holdout.


In [ ]:
calibration_rows = []

for name in trained_models:
    temp = pd.DataFrame({
        "probability": probabilities[name].values,
        "actual": y_test.values,
    })

    temp["probability_bin"] = pd.cut(
        temp["probability"],
        bins=np.linspace(0, 1, 11),
        include_lowest=True,
    )

    grouped = (
        temp.groupby(
            "probability_bin",
            observed=False
        )
        .agg(
            observations=("actual", "count"),
            mean_predicted_probability=("probability", "mean"),
            realized_positive_rate=("actual", "mean"),
        )
        .reset_index()
    )

    grouped["model"] = name
    calibration_rows.append(grouped)

calibration_df = pd.concat(
    calibration_rows,
    ignore_index=True
)

display(calibration_df.head(30))

calibration_df.to_csv(
    TABLE_DIR / "sp500_ml_probability_calibration.csv",
    index=False
)


## 19. Calibration Plot for Best ROC-AUC Model

In [ ]:
best_ml_model = metrics.iloc[0]["model"]

best_calibration = calibration_df[
    calibration_df["model"] == best_ml_model
].dropna(
    subset=[
        "mean_predicted_probability",
        "realized_positive_rate"
    ]
)

fig = plt.figure(figsize=(8, 8))

plt.plot(
    best_calibration["mean_predicted_probability"],
    best_calibration["realized_positive_rate"],
    marker="o",
    label=best_ml_model
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration"
)

plt.title(f"Probability Calibration — {best_ml_model}")
plt.xlabel("Mean Predicted Probability")
plt.ylabel("Realized Positive Rate")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_ml_probability_calibration.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 20. Feature Importance — Tree Models

In [ ]:
feature_importance_tables = {}

for name in ["Random Forest", "XGBoost"]:
    if name not in trained_models:
        continue

    pipeline_or_model = trained_models[name]

    if hasattr(pipeline_or_model, "named_steps"):
        fitted_model = pipeline_or_model.named_steps["model"]
    else:
        fitted_model = pipeline_or_model

    if hasattr(fitted_model, "feature_importances_"):
        importance = pd.DataFrame({
            "feature": feature_columns,
            "importance": fitted_model.feature_importances_,
        }).sort_values(
            "importance",
            ascending=False
        )

        feature_importance_tables[name] = importance

        print("=" * 72)
        print(name)
        print("=" * 72)
        display(importance.head(20))

        safe_name = (
            name.lower()
            .replace(" ", "_")
        )

        importance.to_csv(
            TABLE_DIR / f"sp500_{safe_name}_feature_importance.csv",
            index=False
        )


## 21. Plot Random Forest Feature Importance

In [ ]:
if "Random Forest" in feature_importance_tables:
    importance = feature_importance_tables["Random Forest"].head(15)

    fig = plt.figure(figsize=(10, 7))

    plt.barh(
        importance["feature"].iloc[::-1],
        importance["importance"].iloc[::-1]
    )

    plt.title("Random Forest — Top Feature Importances")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()

    path = FIGURE_DIR / "sp500_random_forest_feature_importance.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Saved: {path}")
else:
    print("Random Forest importance unavailable.")


## 22. Logistic Regression Coefficients

In [ ]:
if "Logistic Regression" in trained_models:
    logistic_pipeline = trained_models["Logistic Regression"]

    logistic_model = logistic_pipeline.named_steps["model"]

    coefficients = pd.DataFrame({
        "feature": feature_columns,
        "coefficient": logistic_model.coef_[0],
    })

    coefficients["absolute_coefficient"] = (
        coefficients["coefficient"].abs()
    )

    coefficients = coefficients.sort_values(
        "absolute_coefficient",
        ascending=False
    )

    display(coefficients.head(20))

    coefficients.to_csv(
        TABLE_DIR / "sp500_logistic_regression_coefficients.csv",
        index=False
    )


## 23. Machine Learning Probability Dataset

Predictions are stored alongside the test dates for later strategy research and backtesting.

No test predictions are written back into the raw master dataset.


In [ ]:
prediction_output = test_df[
    ["Date", "Close", "next_day_return", "target"]
].copy()

for name in trained_models:
    safe_name = (
        name.lower()
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
    )

    prediction_output[f"{safe_name}_prediction"] = (
        predictions[name].values
    )

    prediction_output[f"{safe_name}_probability_up"] = (
        probabilities[name].values
    )

prediction_path = INTERIM_DIR / "sp500_ml_test_predictions.parquet"

prediction_output.to_parquet(
    prediction_path,
    index=False
)

print(f"Saved: {prediction_path}")
display(prediction_output.head())


## 24. Probability Threshold Research

For the strongest probability model, evaluate several probability thresholds.

This is descriptive threshold analysis. The threshold is not selected using future performance beyond the holdout.


In [ ]:
threshold_rows = []

best_probability = probabilities[best_ml_model]

for threshold in [0.50, 0.55, 0.60, 0.65, 0.70]:
    threshold_prediction = (
        best_probability >= threshold
    ).astype(int)

    threshold_rows.append({
        "model": best_ml_model,
        "threshold": threshold,
        "signal_rate": threshold_prediction.mean(),
        "precision": precision_score(
            y_test,
            threshold_prediction,
            zero_division=0
        ),
        "recall": recall_score(
            y_test,
            threshold_prediction,
            zero_division=0
        ),
        "f1": f1_score(
            y_test,
            threshold_prediction,
            zero_division=0
        ),
    })

threshold_analysis = pd.DataFrame(threshold_rows)

display(threshold_analysis)

threshold_analysis.to_csv(
    TABLE_DIR / "sp500_ml_probability_threshold_analysis.csv",
    index=False
)


## 25. Prediction Probability Timeline

In [ ]:
fig = plt.figure(figsize=(14, 6))

plt.plot(
    test_df["Date"],
    best_probability.values
)

plt.axhline(
    0.50,
    linestyle="--",
    label="0.50 threshold"
)

plt.title(f"{best_ml_model} — Probability of Positive Next-Day Return")
plt.xlabel("Date")
plt.ylabel("Probability")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_best_ml_probability_timeline.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 26. Compare ML Models with Baseline Direction

The always-up classifier is included as a simple directional benchmark because the S&P 500 can have a positive unconditional frequency of up days.

This benchmark is not a trading strategy.


In [ ]:
always_up = np.ones(len(y_test), dtype=int)

always_up_metrics = {
    "model": "Always Up Baseline",
    "accuracy": accuracy_score(y_test, always_up),
    "precision": precision_score(
        y_test, always_up, zero_division=0
    ),
    "recall": recall_score(
        y_test, always_up, zero_division=0
    ),
    "f1": f1_score(
        y_test, always_up, zero_division=0
    ),
    "roc_auc": np.nan,
}

metrics_with_baseline = pd.concat(
    [
        metrics,
        pd.DataFrame([always_up_metrics]),
    ],
    ignore_index=True,
)

display(metrics_with_baseline)

metrics_with_baseline.to_csv(
    TABLE_DIR / "sp500_ml_models_with_directional_baseline.csv",
    index=False
)


## 27. Machine Learning Research Report

In [ ]:
ml_report = {
    "dataset": {
        "rows": int(len(df)),
        "modeling_rows": int(len(model_df)),
        "start": df["Date"].min().strftime("%Y-%m-%d"),
        "end": df["Date"].max().strftime("%Y-%m-%d"),
    },
    "split": {
        "train_rows": int(len(train_df)),
        "test_rows": int(len(test_df)),
        "train_end": train_df["Date"].max().strftime("%Y-%m-%d"),
        "test_start": test_df["Date"].min().strftime("%Y-%m-%d"),
        "test_end": test_df["Date"].max().strftime("%Y-%m-%d"),
    },
    "features": feature_columns,
    "models": metrics_with_baseline.to_dict(
        orient="records"
    ),
    "best_model_by_roc_auc": best_ml_model,
    "xgboost_available": xgb_available,
    "methodological_note": (
        "Features are constructed from current and historical observations. "
        "The target is the next-day return direction. The split is chronological. "
        "No random shuffling is used. The holdout results are research baselines "
        "and require walk-forward validation before any trading interpretation."
    ),
}

report_path = REPORT_DIR / "sp500_machine_learning_models_report.json"

report_path.write_text(
    json.dumps(ml_report, indent=2, default=str),
    encoding="utf-8"
)

print(json.dumps(ml_report, indent=2, default=str))
print(f"\nSaved: {report_path}")


## 28. Save Model Input Dataset

In [ ]:
model_input_path = INTERIM_DIR / "sp500_machine_learning_inputs.parquet"

model_df.to_parquet(
    model_input_path,
    index=False
)

print(f"Saved: {model_input_path}")


## 29. Final Raw Dataset Integrity Check

In [ ]:
master_check = pd.read_csv(
    MASTER_PATH,
    low_memory=False
)

assert list(master_check.columns) == EXPECTED_COLUMNS
assert len(master_check) == len(df)

master_dates = pd.to_datetime(
    master_check["Date"],
    errors="coerce"
)

assert master_dates.notna().all()
assert master_dates.is_unique
assert master_dates.is_monotonic_increasing

print("Raw master dataset integrity after machine learning: PASS")
print(f"Master rows: {len(master_check):,}")


# Notebook 08 Complete

Notebook 08 has established supervised machine-learning baselines for next-day S&P 500 return direction.

### Models
- Logistic Regression
- Random Forest
- HistGradientBoosting
- XGBoost when available

### Outputs
- Accuracy
- Precision
- Recall
- F1
- ROC-AUC
- Classification reports
- Confusion matrices
- Probability calibration
- Feature importance
- Logistic coefficients
- Probability-threshold analysis
- Test probability predictions
- Machine-learning research report

### Important methodological boundary

The chronological holdout is a first evaluation layer. It is **not sufficient** to establish a deployable trading edge.

The next stage should evaluate models with walk-forward / expanding-window validation and trading-aware metrics.

**Next notebook:** Notebook 09 — Deep Learning Models.

Run Notebook 08 from top to bottom and verify the outputs before proceeding.
